# Data exploration — first 100 rows of every table

Queries the local Parquet lake in place through **DuckDB** (no import step). Each
table is a Hive-partitioned Parquet dataset (`region=<code>/data.parquet`) or a
single file; `hive_partitioning=1` reconstructs the `region` column from the path.

Run top-to-bottom. `head('<name>')` returns the first 100 rows of any table.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_columns', None, 'display.width', 220)

# resolve the repo root (dir containing `data/`) by walking up from the cwd
PROJECT = Path.cwd()
while PROJECT != PROJECT.parent and not (PROJECT / 'data').is_dir():
    PROJECT = PROJECT.parent

con = duckdb.connect()

# friendly name -> Parquet glob (relative to PROJECT)
TABLES = {
    # -- raw lake --
    'raw_molit_resale':    'data/raw/molit_resale/region=*/data.parquet',       # label (분양권+입주권)
    'raw_molit_apt_trade': 'data/raw/molit_apt_trade/region=*/data.parquet',    # comps (아파트 매매)
    'raw_commercial':      'data/raw/commercial/region=*/data.parquet',         # 상가 POI
    'raw_applyhome':       'data/raw/applyhome/data.parquet',                   # 청약홈 분양정보
    'raw_ecos_macro':      'data/raw/ecos_macro/data.parquet',                  # 한은 macro (monthly)
    'raw_schools':         'data/raw/schools/data.parquet',                     # NEIS 학교
    # -- processed / geocoded --
    'geo_molit_resale':    'data/processed/molit_resale/region=*/data.parquet',    # label + lat/lon + precision
    'geo_molit_apt_trade': 'data/processed/molit_apt_trade/region=*/data.parquet', # comps + lat/lon + precision
    'proc_commercial':     'data/processed/commercial/region=*/data.parquet',      # 상가 cleaned
    'geo_schools':         'data/processed/schools/geocoded.parquet',              # schools + lat/lon
    'geocode_cache':       'data/processed/geocode/cache.parquet',                 # query -> coord cache
}


def head(name: str, n: int = 100) -> pd.DataFrame:
    """First `n` rows of a lake table via DuckDB."""
    path = str(PROJECT / TABLES[name])
    return con.execute(
        f"SELECT * FROM read_parquet('{path}', hive_partitioning=1) LIMIT {n}"
    ).df()


PROJECT = /Users/kibeom/Desktop/coding/korea_property_value


## Overview — row & column counts per table

In [2]:
rows = []
for name, glob in TABLES.items():
    path = str(PROJECT / glob)
    q = f"read_parquet('{path}', hive_partitioning=1)"
    n_rows = con.execute(f'SELECT count(*) FROM {q}').fetchone()[0]
    n_cols = len(con.execute(f'SELECT * FROM {q} LIMIT 0').df().columns)
    rows.append({'table': name, 'rows': n_rows, 'cols': n_cols})
pd.DataFrame(rows)

,table,rows,cols
0,raw_molit_resale,214399,17
1,raw_molit_apt_trade,2139329,22
2,raw_commercial,1116244,40
3,raw_applyhome,6662,12
4,raw_ecos_macro,127,5
5,raw_schools,4077,9
6,geo_molit_resale,214399,20
7,geo_molit_apt_trade,2139329,26
8,proc_commercial,1116244,19
9,geo_schools,4077,12


## Raw lake

### `raw_molit_resale` — training label (분양권 + 입주권 전매 실거래가)

In [3]:
head('raw_molit_resale')

,region_code,dong,jibun,complex_name,exclusive_area_m2,floor,deal_year,deal_month,deal_day,price_manwon,right_type,deal_channel,is_cancelled,deal_date,price_per_m2,deal_ym,region
0,11110,숭인동,1419-2,에비뉴청계Ⅰ,16.23,7,2023,12,20,23400,분양권,중개거래,False,2023-12-20,1441.774492,202312,11110
1,11110,교남동,62-1,경희궁자이(2BL),84.94,4,2016,1,26,78249,분양권,None,False,2016-01-26,921.226748,201601,11110
2,11110,교남동,BL-3,경희궁자이(3BL),59.85,7,2016,1,23,70000,입주권,None,False,2016-01-23,1169.590643,201601,11110
3,11110,교남동,62-1,경희궁자이(2BL),84.61,12,2016,1,15,80327,분양권,None,False,2016-01-15,949.379506,201601,11110
4,11110,교남동,62-1,경희궁자이(2BL),84.94,9,2016,1,28,81581,입주권,None,False,2016-01-28,960.454438,201601,11110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,11110,교남동,62-1,경희궁자이(2BL),84.83,17,2016,5,12,80607,분양권,None,False,2016-05-12,950.218083,201605,11110
96,11110,교남동,BL-3,경희궁자이(3BL),77.65,11,2016,6,9,80000,입주권,None,False,2016-06-09,1030.264005,201606,11110
97,11110,교남동,62-1,경희궁자이(2BL),101.99,8,2016,6,14,107000,입주권,None,False,2016-06-14,1049.122463,201606,11110
98,11110,교남동,62-1,경희궁자이(2BL),84.93,5,2016,6,29,79721,분양권,None,False,2016-06-29,938.667138,201606,11110


### `raw_molit_apt_trade` — comparable sales (아파트 매매 실거래가)

In [4]:
head('raw_molit_apt_trade')

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,dealingGbn,estateAgentSggNm,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm,deal_ym,region
0,None,창림,1994,None,None,None,"20,000",29,1,2016,None,None,59.67,1,639-20,N,None,11110,None,창신동,201601,11110
1,None,창신쌍용1,1992,None,None,None,"30,800",16,1,2016,None,None,54.7,13,702,N,None,11110,None,창신동,201601,11110
2,None,아남1,1995,None,None,None,"64,500",20,1,2016,None,None,84.9,20,4,N,None,11110,None,명륜2가,201601,11110
3,None,아남1,1995,None,None,None,"54,800",12,1,2016,None,None,84.9,1,4,N,None,11110,None,명륜2가,201601,11110
4,None,효성쥬얼리시티,2006,None,None,None,"47,800",23,1,2016,None,None,84.7,14,48-2,N,None,11110,None,인의동,201601,11110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,None,아남1,1995,None,None,None,"63,000",5,2,2016,None,None,84.9,7,4,N,None,11110,None,명륜2가,201602,11110
96,None,광화문스페이스본(101동~105동),2008,None,None,None,"82,500",23,2,2016,None,None,94.51,11,9,N,None,11110,None,사직동,201602,11110
97,None,삼전솔하임2차,2012,None,None,None,"11,700",16,2,2016,None,None,16.67,6,296-19,N,None,11110,None,숭인동,201602,11110
98,None,현대,2000,None,None,None,"59,790",4,2,2016,None,None,114.9,8,82,N,None,11110,None,무악동,201602,11110


### `raw_commercial` — 상가(상권)정보 POIs (built-in lat/lon)

In [4]:
head('raw_commercial')

,bizesId,bizesNm,brchNm,indsLclsCd,indsLclsNm,indsMclsCd,indsMclsNm,indsSclsCd,indsSclsNm,ksicCd,ksicNm,ctprvnCd,ctprvnNm,signguCd,signguNm,adongCd,adongNm,ldongCd,ldongNm,lnoCd,plotSctCd,plotSctNm,lnoMnno,lnoSlno,lnoAdr,rdnmCd,rdnm,bldMnno,bldSlno,bldMngNo,bldNm,rdnmAdr,oldZipcd,newZipcd,dongNo,flrNo,hoNo,lon,lat,region
0,MA010120220700034603,치휴한방병원,,Q1,보건의료,Q101,병원,Q10104,한방병원,Q86104,한방병원,11,서울특별시,11110,종로구,11110640,이화동,1111016600,연건동,1111016600100390000,1,대지,39,,서울특별시 종로구 연건동 39,111103100002,서울특별시 종로구 대학로,89,,1111016600100390000010801,1~2층지상,서울특별시 종로구 대학로 89,110500,03082,,,,127.001811349613,37.5792328193644,11110
1,MA010120220800000084,고향집,,I2,음식,I201,한식,I20101,백반/한정식,I56111,한식 일반 음식점업,11,서울특별시,11110,종로구,11110615,종로1.2.3.4가동,1111015400,장사동,1111015400100800000,1,대지,80,,서울특별시 종로구 장사동 80,111104100054,서울특별시 종로구 돈화문로2길,25,11,1111015400100800000000002,,서울특별시 종로구 돈화문로2길 25-11,110430,03193,,,,126.993830863432,37.5698623269561,11110
2,MA010120220800000099,청안,,G2,소매,G209,섬유·의복·신발 소매,G20902,여성 의류 소매업,G47412,여자용 겉옷 소매업,11,서울특별시,11110,종로구,11110540,삼청동,1111014200,소격동,1111014200100610000,1,대지,61,,서울특별시 종로구 소격동 61,111103100007,서울특별시 종로구 삼청로,48,12,1111014200100610000025897,,서울특별시 종로구 삼청로 48-12,110230,03053,,1,,126.980584283332,37.5804385683911,11110
3,MA010120220800000112,수피아,,G2,소매,G217,시계·귀금속 소매,G21701,시계/귀금속 소매업,G47830,시계 및 귀금속 소매업,11,서울특별시,11110,종로구,11110615,종로1.2.3.4가동,1111015600,종로3가,1111015600100270000,1,대지,27,,서울특별시 종로구 종로3가 27,111103100013,서울특별시 종로구 종로,135,,1111015600100260000000001,우리귀금속상가,서울특별시 종로구 종로 135,110123,03138,,,,126.992472433488,37.570736213665,11110
4,MA010120220800000119,광명피아노,,G2,소매,G212,기타 생활용품 소매,G21203,악기 소매업,G47593,악기 소매업,11,서울특별시,11110,종로구,11110615,종로1.2.3.4가동,1111013700,낙원동,1111013700102880000,1,대지,288,,서울특별시 종로구 낙원동 288,111102100001,서울특별시 종로구 삼일대로,428,,1111013700102540004016150,낙원상가,서울특별시 종로구 삼일대로 428,110320,03140,,2,,126.987650017666,37.5722224497393,11110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,MA010120220800001884,가야재,,G2,소매,G218,장식품 소매,G21801,예술품 소매업,G47841,예술품 및 골동품 소매업,11,서울특별시,11110,종로구,11110615,종로1.2.3.4가동,1111012800,관훈동,1111012800101920001,1,대지,192,1,서울특별시 종로구 관훈동 192-1,111104100259,서울특별시 종로구 인사동길,39,1,1111012800101920001014029,,서울특별시 종로구 인사동길 39-1,110300,03145,,,,126.984864985998,37.5739159960281,11110
96,MA010120220800001897,락궁,,I2,음식,I202,중식,I20201,중국집,I56121,중식 음식점업,11,서울특별시,11110,종로구,11110560,평창동,1111018300,평창동,1111018300103390000,1,대지,339,,서울특별시 종로구 평창동 339,111104100487,서울특별시 종로구 평창11길,5,,1111018300103390000023108,,서울특별시 종로구 평창11길 5,110847,03008,,,,126.966602512272,37.6062038064802,11110
97,MA010120220800001900,벨&보,,S2,수리·개인,S207,이용·미용,S20701,미용실,S96112,두발 미용업,11,서울특별시,11110,종로구,11110580,교남동,1111018000,교북동,1111018000100010011,1,대지,1,11,서울특별시 종로구 교북동 1-11,111103000008,서울특별시 종로구 통일로,224,1,1111018000100010011020367,,서울특별시 종로구 통일로 224-1,110100,03029,,,,126.95985935434,37.5729190475213,11110
98,MA010120220800001946,광진전자,,S2,수리·개인,S205,가전제품 수리,S20501,가전제품 수리업,S95310,가전제품 수리업,11,서울특별시,11110,종로구,11110615,종로1.2.3.4가동,1111015400,장사동,1111015400101160004,1,대지,116,4,서울특별시 종로구 장사동 116-4,111103100021,서울특별시 종로구 청계천로,159,,1111015400101160004015909,세운상가,서울특별시 종로구 청계천로 159,110430,03194,,4,,126.995234520229,37.5692857548513,11110


### `raw_applyhome` — 청약홈 분양정보 (enrichment: 분양가, 입주예정, 세대수, 건설사)

In [5]:
head('raw_applyhome')

,pblanc_no,house_name,house_kind,supply_region,address,notice_date,move_in_ym,total_units,builder,house_type,exclusive_area_m2,supply_price_manwon
0,2026000355,월계 중흥S-클래스 리비에르,APT,서울,서울특별시 노원구 월계동 487-17번지 일대,2026-07-16,202905,135,중흥토건(주),036.9533,36.95,69590
1,2026000355,월계 중흥S-클래스 리비에르,APT,서울,서울특별시 노원구 월계동 487-17번지 일대,2026-07-16,202905,135,중흥토건(주),059.9667A,59.96,126350
2,2026000355,월계 중흥S-클래스 리비에르,APT,서울,서울특별시 노원구 월계동 487-17번지 일대,2026-07-16,202905,135,중흥토건(주),059.9424B,59.94,124200
3,2026000355,월계 중흥S-클래스 리비에르,APT,서울,서울특별시 노원구 월계동 487-17번지 일대,2026-07-16,202905,135,중흥토건(주),084.9807A,84.98,149910
4,2026000336,평택역 더센트럴45,APT,경기,경기도 평택시 평택동 45-1,2026-07-09,202608,99,파인건설 주식회사,079.9213,79.92,55000
...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026000275,장위 푸르지오 마크원,APT,서울,서울특별시 성북구 장위동 68-37 일대,2026-06-19,203009,1032,(주)대우건설,084.7500D,84.75,176570
96,2026000275,장위 푸르지오 마크원,APT,서울,서울특별시 성북구 장위동 68-37 일대,2026-06-19,203009,1032,(주)대우건설,101.7400,101.74,194570
97,2026000275,장위 푸르지오 마크원,APT,서울,서울특별시 성북구 장위동 68-37 일대,2026-06-19,203009,1032,(주)대우건설,114.2100,114.21,217140
98,2026000258,고덕국제신도시 수자인하우스디,APT,경기,경기도 평택시 고덕국제화계획지구 A-67블록,2026-06-11,202902,403,대보건설(주),084.8390A,84.83,56980


### `raw_ecos_macro` — 한국은행 monthly macro (base rate, mortgage, M2, CSI)

In [9]:
df_ecos_raw = head('raw_ecos_macro')
df_ecos_raw.set_index('deal_ym')

,base_rate,m2,mortgage_rate,sentiment_csi
deal_ym,,,,
201601,1.5,2108840.8,3.10,100.4
201602,1.5,2135969.9,2.99,98.1
201603,1.5,2132190.5,2.97,100.4
201604,1.5,2145161.1,2.93,102.1
201605,1.5,2161918.8,2.89,99.6
...,...,...,...,...
202312,3.5,3747579.6,4.16,99.5
202401,3.5,3725946.1,3.99,101.5
202402,3.5,3758300.3,3.96,101.7


### `raw_schools` — NEIS 학교기본정보 (도로명주소 only)

In [8]:
head('raw_schools')

,school_code,name,school_type,foundation,office,sido,road_address,road_address_detail,coedu
0,7010057,가락고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 송파구 송이로 42,"(송파동,가락고등학교)",남여공학
1,7130165,가락중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 송파구 송이로 45,"(송파동,가락중학교)",남여공학
2,7041164,가산중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 금천구 시흥대로115길 48,(독산동),남여공학
3,7130166,가원중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 송파구 중대로10길 40-18,"(가락동,가원중학교)",남여공학
4,7011169,가재울고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 서대문구 수색로 100-35,(북가좌동),남여공학
...,...,...,...,...,...,...,...,...,...
95,7011111,구암고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 관악구 성현로 57,"(봉천동,구암고등학교)",남여공학
96,7132127,구암중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 관악구 성현로 53,(봉천동),남여공학
97,7134127,구의중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 광진구 광나루로30길 80,(화양동),남여공학
98,7010069,구일고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 구로구 구일로 90-51,(구로동),남여공학


## Processed / geocoded

`geocode_precision` tiers: `parcel` / `borrow` / `keyword` = building-level, `centroid` = dong fallback (flagged).

### `geo_molit_resale` — label + lat/lon + geocode_precision

In [9]:
head('geo_molit_resale')

,region_code,dong,jibun,complex_name,exclusive_area_m2,floor,deal_year,deal_month,deal_day,price_manwon,right_type,deal_channel,is_cancelled,deal_date,price_per_m2,deal_ym,lat,lon,geocode_precision,region
0,11110,숭인동,1419-2,에비뉴청계Ⅰ,16.23,7,2023,12,20,23400,분양권,중개거래,False,2023-12-20,1441.774492,202312,37.573239,127.022097,parcel,11110
1,11110,교남동,62-1,경희궁자이(2BL),84.94,4,2016,1,26,78249,분양권,None,False,2016-01-26,921.226748,201601,37.570697,126.963608,borrow,11110
2,11110,교남동,BL-3,경희궁자이(3BL),59.85,7,2016,1,23,70000,입주권,None,False,2016-01-23,1169.590643,201601,37.570697,126.963608,borrow,11110
3,11110,교남동,62-1,경희궁자이(2BL),84.61,12,2016,1,15,80327,분양권,None,False,2016-01-15,949.379506,201601,37.570697,126.963608,borrow,11110
4,11110,교남동,62-1,경희궁자이(2BL),84.94,9,2016,1,28,81581,입주권,None,False,2016-01-28,960.454438,201601,37.570697,126.963608,borrow,11110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,11110,교남동,62-1,경희궁자이(2BL),84.83,17,2016,5,12,80607,분양권,None,False,2016-05-12,950.218083,201605,37.570697,126.963608,borrow,11110
96,11110,교남동,BL-3,경희궁자이(3BL),77.65,11,2016,6,9,80000,입주권,None,False,2016-06-09,1030.264005,201606,37.570697,126.963608,borrow,11110
97,11110,교남동,62-1,경희궁자이(2BL),101.99,8,2016,6,14,107000,입주권,None,False,2016-06-14,1049.122463,201606,37.570697,126.963608,borrow,11110
98,11110,교남동,62-1,경희궁자이(2BL),84.93,5,2016,6,29,79721,분양권,None,False,2016-06-29,938.667138,201606,37.570697,126.963608,borrow,11110


### `geo_molit_apt_trade` — comps + lat/lon + geocode_precision

In [12]:
df_molit_apt_trade = head('geo_molit_apt_trade')
df_molit_apt_trade

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,dealingGbn,estateAgentSggNm,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm,deal_ym,deal_date,lat,lon,geocode_precision,region
0,None,창림,1994,None,None,None,"20,000",29,1,2016,None,None,59.67,1,639-20,N,None,11110,None,창신동,201601,2016-01-29,37.576000,127.009360,parcel,11110
1,None,창신쌍용1,1992,None,None,None,"30,800",16,1,2016,None,None,54.7,13,702,N,None,11110,None,창신동,201601,2016-01-16,37.580609,127.013926,parcel,11110
2,None,아남1,1995,None,None,None,"64,500",20,1,2016,None,None,84.9,20,4,N,None,11110,None,명륜2가,201601,2016-01-20,37.585614,126.999342,parcel,11110
3,None,아남1,1995,None,None,None,"54,800",12,1,2016,None,None,84.9,1,4,N,None,11110,None,명륜2가,201601,2016-01-12,37.585614,126.999342,parcel,11110
4,None,효성쥬얼리시티,2006,None,None,None,"47,800",23,1,2016,None,None,84.7,14,48-2,N,None,11110,None,인의동,201601,2016-01-23,37.571558,126.998718,parcel,11110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,None,아남1,1995,None,None,None,"63,000",5,2,2016,None,None,84.9,7,4,N,None,11110,None,명륜2가,201602,2016-02-05,37.585614,126.999342,parcel,11110
96,None,광화문스페이스본(101동~105동),2008,None,None,None,"82,500",23,2,2016,None,None,94.51,11,9,N,None,11110,None,사직동,201602,2016-02-23,37.574359,126.968850,parcel,11110
97,None,삼전솔하임2차,2012,None,None,None,"11,700",16,2,2016,None,None,16.67,6,296-19,N,None,11110,None,숭인동,201602,2016-02-16,37.572354,127.016733,parcel,11110
98,None,현대,2000,None,None,None,"59,790",4,2,2016,None,None,114.9,8,82,N,None,11110,None,무악동,201602,2016-02-04,37.575631,126.961060,parcel,11110


### `proc_commercial` — cleaned 상가 amenity table

In [13]:
head('proc_commercial')

,bizesId,bizesNm,brchNm,indsLclsCd,indsLclsNm,indsMclsCd,indsMclsNm,indsSclsCd,indsSclsNm,ksicCd,ksicNm,signguCd,ldongCd,adongCd,lnoAdr,rdnmAdr,lon,lat,region
0,MA010120220700034603,치휴한방병원,None,Q1,보건의료,Q101,병원,Q10104,한방병원,Q86104,한방병원,11110,1111016600,11110640,서울특별시 종로구 연건동 39,서울특별시 종로구 대학로 89,127.001811,37.579233,11110
1,MA010120220800000084,고향집,None,I2,음식,I201,한식,I20101,백반/한정식,I56111,한식 일반 음식점업,11110,1111015400,11110615,서울특별시 종로구 장사동 80,서울특별시 종로구 돈화문로2길 25-11,126.993831,37.569862,11110
2,MA010120220800000099,청안,None,G2,소매,G209,섬유·의복·신발 소매,G20902,여성 의류 소매업,G47412,여자용 겉옷 소매업,11110,1111014200,11110540,서울특별시 종로구 소격동 61,서울특별시 종로구 삼청로 48-12,126.980584,37.580439,11110
3,MA010120220800000112,수피아,None,G2,소매,G217,시계·귀금속 소매,G21701,시계/귀금속 소매업,G47830,시계 및 귀금속 소매업,11110,1111015600,11110615,서울특별시 종로구 종로3가 27,서울특별시 종로구 종로 135,126.992472,37.570736,11110
4,MA010120220800000119,광명피아노,None,G2,소매,G212,기타 생활용품 소매,G21203,악기 소매업,G47593,악기 소매업,11110,1111013700,11110615,서울특별시 종로구 낙원동 288,서울특별시 종로구 삼일대로 428,126.987650,37.572222,11110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,MA010120220800001884,가야재,None,G2,소매,G218,장식품 소매,G21801,예술품 소매업,G47841,예술품 및 골동품 소매업,11110,1111012800,11110615,서울특별시 종로구 관훈동 192-1,서울특별시 종로구 인사동길 39-1,126.984865,37.573916,11110
96,MA010120220800001897,락궁,None,I2,음식,I202,중식,I20201,중국집,I56121,중식 음식점업,11110,1111018300,11110560,서울특별시 종로구 평창동 339,서울특별시 종로구 평창11길 5,126.966603,37.606204,11110
97,MA010120220800001900,벨&보,None,S2,수리·개인,S207,이용·미용,S20701,미용실,S96112,두발 미용업,11110,1111018000,11110580,서울특별시 종로구 교북동 1-11,서울특별시 종로구 통일로 224-1,126.959859,37.572919,11110
98,MA010120220800001946,광진전자,None,S2,수리·개인,S205,가전제품 수리,S20501,가전제품 수리업,S95310,가전제품 수리업,11110,1111015400,11110615,서울특별시 종로구 장사동 116-4,서울특별시 종로구 청계천로 159,126.995235,37.569286,11110


### `geo_schools` — schools + lat/lon

In [12]:
head('geo_schools')

,school_code,name,school_type,foundation,office,sido,road_address,road_address_detail,coedu,lat,lon,geocode_precision
0,7010057,가락고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 송파구 송이로 42,"(송파동,가락고등학교)",남여공학,37.501356,127.116219,road
1,7130165,가락중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 송파구 송이로 45,"(송파동,가락중학교)",남여공학,37.502449,127.117795,road
2,7041164,가산중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 금천구 시흥대로115길 48,(독산동),남여공학,37.467937,126.894520,road
3,7130166,가원중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 송파구 중대로10길 40-18,"(가락동,가원중학교)",남여공학,37.490968,127.122979,road
4,7011169,가재울고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 서대문구 수색로 100-35,(북가좌동),남여공학,37.574343,126.909567,road
...,...,...,...,...,...,...,...,...,...,...,...,...
95,7011111,구암고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 관악구 성현로 57,"(봉천동,구암고등학교)",남여공학,37.492460,126.949783,road
96,7132127,구암중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 관악구 성현로 53,(봉천동),남여공학,37.492766,126.949036,road
97,7134127,구의중학교,중학교,공립,서울특별시교육청,서울특별시,서울특별시 광진구 광나루로30길 80,(화양동),남여공학,37.543173,127.078659,road
98,7010069,구일고등학교,고등학교,공립,서울특별시교육청,서울특별시,서울특별시 구로구 구일로 90-51,(구로동),남여공학,37.494385,126.874524,road


### `geocode_cache` — query → coord cache (method: vworld_parcel / vworld_road / kakao_keyword)

In [14]:
head('geocode_cache')

,method,query,status,lat,lon
0,vworld_road,서울특별시 송파구 송이로 42,OK,37.501356,127.116219
1,vworld_road,서울특별시 송파구 송이로 45,OK,37.502449,127.117795
2,vworld_road,서울특별시 금천구 시흥대로115길 48,OK,37.467937,126.894520
3,vworld_road,서울특별시 송파구 중대로10길 40-18,OK,37.490968,127.122979
4,vworld_road,서울특별시 서대문구 수색로 100-35,OK,37.574343,126.909567
...,...,...,...,...,...
95,vworld_road,서울특별시 금천구 문성로 67,OK,37.478228,126.910462
96,vworld_road,서울특별시 관악구 난곡로16길 32,OK,37.466416,126.922411
97,vworld_road,서울특별시 관악구 난곡로34길 80,OK,37.471447,126.924852
98,vworld_road,서울특별시 성북구 한천로 660-30,OK,37.618256,127.056120
